
# Products Data Processing


### Imports

In [0]:
from pyspark.sql.functions import *


### Schema setup for Bronze, Silver and Gold layers

In [0]:
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'


### Defining widgets for runtime configuration
#### Syntax: dbutils.widgets.text(internal handle, pre-filled value, display text)
#### /*.csv = 'read every csv file in this folder'

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'products', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(catalog)
print(data_source)

base_path = f's3://databricks-data-engineering-project/{data_source}/*.csv'
print(base_path)

fmcg
products
s3://databricks-data-engineering-project/products/*.csv


### Read CSV from source

In [0]:
df = (
    spark.read.format('csv')
    .option('header', True)
    .option('inferSchema', True)
    .load(base_path)
    .withColumn('read_timestamp', current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)

display(df.limit(10))

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-10-23T12:41:56.299Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-10-23T12:41:56.299Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-10-23T12:41:56.299Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-10-23T12:41:56.299Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-10-23T12:41:56.299Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-10-23T12:41:56.299Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-10-23T12:41:56.299Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-10-23T12:41:56.299Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-10-23T12:41:56.299Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-10-23T12:41:56.299Z,products.csv,1388


### Inspect the inferred schema

In [0]:
df.printSchema()

root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



### Ingesting raw data into Bronze layer

In [0]:
(
    df.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')
)

### Silver Processing

In [0]:
df_bronze = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source}')
display(df_bronze.limit(10))

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-10-23T12:42:03.501Z,products.csv,1388


### Check for duplicate products

In [0]:
display(spark.sql(f"""
SELECT
    product_id,
    COUNT(product_id) AS duplicate_count
FROM
    {catalog}.{bronze_schema}.{data_source}
GROUP BY
    product_id
HAVING
    COUNT(product_id) > 1;
"""))

product_id,duplicate_count
25891101,2
25891102,2


### Deduplicate on `product_id` and report row counts

In [0]:
print(f'Rows before dedupe: {df_bronze.count()}')

df_silver = df_bronze.drop_duplicates(subset = ['product_id'])
print(f'Rows after dedupe: {df_silver.count()}')

Rows before dedupe: 20
Rows after dedupe: 18


### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Distinct `category` names in silver

In [0]:
display(spark.sql(f"""
SELECT DISTINCT
    category
FROM
    {catalog}.{silver_schema}.{data_source};
"""))

category
healthy snacks
energy bars
granola & cereals
recovery dairy
protien bars
electrolyte mix


### Clean and standardize `category` column

In [0]:
df_silver = df_silver.withColumn(
    'category', 
    when(col('category').isNull(), None).otherwise(initcap(col('category')))
)

### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Verify `category` names after cleaning

In [0]:
display(spark.sql(f"""
SELECT DISTINCT
    category
FROM
    {catalog}.{silver_schema}.{data_source};
"""))

category
Healthy Snacks
Energy Bars
Granola & Cereals
Recovery Dairy
Protien Bars
Electrolyte Mix


### Standardize 'protein' spelling in `product_name` and `category` columns

In [0]:
df_silver = (
    df_silver
    .withColumn('product_name', regexp_replace(col('product_name'), '(?i)protien', 'Protein'))
    .withColumn('category', regexp_replace(col('category'), '(?i)protien', 'Protein'))
)

# (?i) Case-insensitive

### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Verify spelling post correction

In [0]:
display(spark.sql(f"""
SELECT
    *
FROM
    {catalog}.{silver_schema}.{data_source};
"""))

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Oats Cookie Bites ChocoChip (180g),25891503,Healthy Snacks,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (80g),25891403,Recovery Dairy,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Oats Cookie Bites ChocoChip (350g),XYZ123,Healthy Snacks,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protein Bars,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),25891601,Electrolyte Mix,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),25891602,Electrolyte Mix,2026-10-23T12:42:03.501Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-10-23T12:42:03.501Z,products.csv,1388


### Add `division` and `variant` columns

In [0]:
df_silver = (
    df_silver.withColumn('division',
        when(col('category') == 'Energy Bars', 'Nutrition Bars')
        .when(col('category') == 'Protein Bars', 'Nutrition Bars')
        .when(col('category') == 'Granola & Cereals', 'Breakfast Foods')
        .when(col('category') == 'Recovery Dairy', 'Dairy & Recovery')
        .when(col('category') == 'Healthy Snacks', 'Healthy Snacks')
        .when(col('category') == 'Electrolyte Mix', 'Hydration & Electrolytes')
        .otherwise(None))
)

df_silver = (
    df_silver.withColumn('variant',
    regexp_extract(col('product_name'), r'\((.*?)\)', 1))
)

# `.*?` lazily captures text up to the next `)` (stops at the first delimiter)
# `1` returns the first capture group — the content inside the parentheses

### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Verify new columns

In [0]:
display(spark.sql(f"""
SELECT
    *
FROM
    {catalog}.{silver_schema}.{data_source};
"""))

product_name,product_id,category,read_timestamp,file_name,file_size,division,variant
SportsBar Oats Cookie Bites ChocoChip (180g),25891503,Healthy Snacks,2026-10-23T12:42:03.501Z,products.csv,1388,Healthy Snacks,180g
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-10-23T12:42:03.501Z,products.csv,1388,Nutrition Bars,40g
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-10-23T12:42:03.501Z,products.csv,1388,Breakfast Foods,300g
SportsBar Greek Yogurt Pro Vanilla (80g),25891403,Recovery Dairy,2026-10-23T12:42:03.501Z,products.csv,1388,Dairy & Recovery,80g
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-10-23T12:42:03.501Z,products.csv,1388,Breakfast Foods,400g
SportsBar Oats Cookie Bites ChocoChip (350g),XYZ123,Healthy Snacks,2026-10-23T12:42:03.501Z,products.csv,1388,Healthy Snacks,350g
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protein Bars,2026-10-23T12:42:03.501Z,products.csv,1388,Nutrition Bars,55g
SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),25891601,Electrolyte Mix,2026-10-23T12:42:03.501Z,products.csv,1388,Hydration & Electrolytes,30 Sachets
SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),25891602,Electrolyte Mix,2026-10-23T12:42:03.501Z,products.csv,1388,Hydration & Electrolytes,15 Sachets
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-10-23T12:42:03.501Z,products.csv,1388,Breakfast Foods,200g


### Clean Product Fields: 
#### 1. Hash `product_code` <br> 2. Sanitize `product_id` <br> 3. Rename `product_name`

In [0]:
df_silver = (
    df_silver.withColumn('product_code', sha2(col('product_name'), 256))
)
# replacing invalid product_ids with a fallback value
# SHA-256 always produces a 64-character hex string, regardless of input length

df_silver = (
    df_silver.withColumn('product_id',
    when(col('product_id').rlike(r'^[0-9]+$'), col('product_id').cast('string'))
    .otherwise(lit(999999).cast('string')))
)
# ^ and $ anchor the match to the start and end of the string, [0-9] matches a single digit, and + accounts for multiple digits

df_silver = (
    df_silver.withColumnRenamed('product_name', 'product')
)

### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Gold Processing - Select final columns

In [0]:
df_gold = df_silver.select('product_code', 'product_id', 'division', 'category', 'product', 'variant')

### Writing staging dimension to gold schema

In [0]:
(
    df_gold.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{gold_schema}.staging_dim_{data_source}')
)

df_gold.show()

+--------------------+----------+--------------------+-----------------+--------------------+----------+
|        product_code|product_id|            division|         category|             product|   variant|
+--------------------+----------+--------------------+-----------------+--------------------+----------+
|e91ba9d665f90254d...|  25891101|      Nutrition Bars|      Energy Bars|SportsBar Energy ...|       60g|
|e92c739a8d78cd6cb...|  25891102|      Nutrition Bars|      Energy Bars|SportsBar Energy ...|       40g|
|102628255d24304d6...|  25891103|      Nutrition Bars|      Energy Bars|SportsBar Energy ...|       25g|
|2e387cef1424d6e7b...|  25891201|      Nutrition Bars|     Protein Bars|SportsBar Protein...|       45g|
|0cb7b2f42657b625f...|  25891202|      Nutrition Bars|     Protein Bars|SportsBar Protein...|       55g|
|889c67757ece9c973...|  25891203|      Nutrition Bars|     Protein Bars|SportsBar Protein...|       65g|
|3cab59f0592428527...|  25891301|     Breakfast Foods|G

### Merge staging dimension with parent

In [0]:
display(spark.sql(f"""
MERGE INTO {catalog}.{gold_schema}.dim_{data_source} AS target
USING {catalog}.{gold_schema}.staging_dim_{data_source} AS source
ON
    target.product_code = source.product_code

WHEN MATCHED THEN
UPDATE
SET
    division = source.division,
    category = source.category,
    product = source.product,
    variant = source.variant

WHEN NOT MATCHED THEN
INSERT (
    product_code,
    division,
    category,
    product,
    variant
)

VALUES (
    source.product_code,
    source.division,
    source.category,
    source.product,
    source.variant
);
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
18,0,0,18
